In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import skew
from scipy.special import boxcox1p

In [3]:
df = pd.read_csv('../data/train.csv')
df = df.drop('Id', axis=1)

In [4]:
def splittingg(df):
    df_num = df.select_dtypes(include=['float64', 'int64'])
    df_cat = df.select_dtypes(include=['object'])

    return df_num, df_cat

In [5]:
def feature_engineering(df):
    df_feat = df.copy()

    # Additional useful features
    df_feat['TotalSF'] = (
        df_feat['TotalBsmtSF'].fillna(0)
        + df_feat['1stFlrSF'].fillna(0)
        + df_feat['2ndFlrSF'].fillna(0)
    )

    df_feat['TotalPorchSF'] = (
        df_feat['OpenPorchSF'].fillna(0)
        + df_feat['EnclosedPorch'].fillna(0)
        + df_feat['3SsnPorch'].fillna(0)
        + df_feat['ScreenPorch'].fillna(0)
    )

    df_feat['TotalBath'] = (
        df_feat['FullBath'].fillna(0)
        + 0.5 * df_feat['HalfBath'].fillna(0)
        + df_feat['BsmtFullBath'].fillna(0)
        + 0.5 * df_feat['BsmtHalfBath'].fillna(0)
    )

    # House age
    df_feat['HouseAge'] = (
        df_feat['YrSold'] - df_feat['YearBuilt']
    )

    df_feat['RemodAge'] = (
        df_feat['YrSold'] - df_feat['YearRemodAdd']
    )

    df_feat['IsRemod'] = (df_feat['YearRemodAdd'] != df_feat['YearBuilt']).astype(int)

    df_feat['IsNew'] = (df_feat['YrSold'] == df_feat['YearBuilt']).astype(int)

    # Missing value handling / cleanup
    df_feat['LotFrontage'] = df_feat['LotFrontage'].fillna(
        df_feat['LotFrontage'].median()
    )
    df_feat = df_feat.drop('GarageYrBlt', axis=1)
    df_feat['MasVnrArea'] = df_feat['MasVnrArea'].fillna(0)

    return df_feat

In [6]:


def handle_skewness(df_feat, threshold=0.75, lam=0.15):

    df_feat = df_feat.copy()

    # Log-transform the target
    if 'SalePrice' in df_feat.columns:
        df_feat['SalePrice'] = np.log1p(df_feat['SalePrice'])

    # Compute skewness of numeric features
    numeric_feats = df_feat.select_dtypes(include=['float64', 'int64']).columns
    skewed_features = df_feat[numeric_feats].apply(lambda x: skew(x.dropna()))

    # Select features above the skewness threshold
    skewed = skewed_features[abs(skewed_features) > threshold].index
    skewed = skewed.drop('SalePrice', errors='ignore')

    # Apply Box-Cox transform
    for col in skewed:
        df_feat[col] = boxcox1p(df_feat[col], lam)

    return df_feat

In [7]:
def handle_categorical(df_cat):
    df_cat = df_cat.copy()

    # Columns where NaN likely means 'None' (absence of feature)
    none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu']
    for col in none_cols:
        if col in df_cat.columns:
            df_cat[col] = df_cat[col].fillna('0')  # Changed from 'None' to '0'

    # Impute remaining categorical missing values with the mode
    for col in df_cat.columns:
        if df_cat[col].isnull().any():
            df_cat[col] = df_cat[col].fillna(df_cat[col].mode()[0])

    # One-hot encode
    df_cat = pd.get_dummies(df_cat, dtype=int)

    return df_cat

In [8]:
def combined(df_feat, df_cat):
    df_final = pd.concat([df_feat, df_cat], axis=1)
    return df_final